<a href="https://colab.research.google.com/github/ImSayvi/Sztuczna-Inteligencja/blob/main/rag_projekt_21198.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q faiss-cpu pymupdf sentence-transformers tqdm ollama
!apt-get update -qq
!apt-get install -y zstd
!curl --http1.1 -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nohup ollama serve > ollama_server.log 2>&1 &
!sleep 10
!ollama pull llama3.2:3b
!ollama list

In [ ]:
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import ollama

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"
LLM_NAME = "llama3.2:3b"

embedder = SentenceTransformer(EMBED_MODEL_NAME)

print("Załadowano model embeddingowy:", EMBED_MODEL_NAME)
print("Model LLM (Ollama):", LLM_NAME)

In [ ]:
DOCS_DIR = "knowledge"
DB_DIR = "faiss_store"

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

print("Katalog roboczy:", os.getcwd())
print("Zawartość:", os.listdir())

In [ ]:
pdf_files = sorted(
    f for f in os.listdir(DOCS_DIR)
    if f.lower().endswith(".pdf")
)

print(f"Znaleziono {len(pdf_files)} plik(ów) PDF w '{DOCS_DIR}':")
for pdf in pdf_files:
    print(" -", pdf)

if not pdf_files:
    raise FileNotFoundError(
        f"Katalog '{DOCS_DIR}' jest pusty. Wgraj do niego pliki PDF "
    )

In [ ]:
embedding_dim = embedder.get_sentence_embedding_dimension()
vector_index = faiss.IndexFlatL2(embedding_dim)
chunk_registry = []

print("Wymiar wektora embeddingu:", embedding_dim)
print("Liczba chunków w indeksie:", vector_index.ntotal)

In [ ]:
class DocumentIndexer:
    """Wczytuje PDF-y, dzieli je na fragmenty i indeksuje w FAISS."""

    def __init__(self, embedder, index, registry, window=900, overlap=150):
        self.embedder = embedder
        self.index = index
        self.registry = registry
        self.window = window
        self.overlap = overlap

    @staticmethod
    def _read_pdf(path):
        doc = pymupdf.open(path)
        pages = []
        for page_num in range(len(doc)):
            raw = doc[page_num].get_text()
            cleaned = " ".join(raw.split())
            pages.append((page_num, cleaned))
        doc.close()
        return pages

    def _split_page(self, page_text):
        fragments = []
        step = max(self.window - self.overlap, 1)
        start = 0
        n = len(page_text)

        while start < n:
            end = start + self.window
            piece = page_text[start:end].strip()
            if piece:
                fragments.append(piece)
            start += step

        return fragments

    def build_chunks(self, pages):
        chunks = []
        for page_num, text in pages:
            if not text:
                continue
            for fragment in self._split_page(text):
                chunks.append((page_num, fragment))
        return chunks

    def index_chunks(self, chunks, source_name):
        vectors = self.embedder.encode(
            [c for _, c in chunks],
            show_progress_bar=True,
            batch_size=32,
        ).astype("float32")

        self.index.add(vectors)

        for local_id, (page_num, text) in enumerate(chunks):
            self.registry.append({
                "source": source_name,
                "page": page_num,
                "local_id": local_id,
                "text": text,
            })

    def save(self, db_dir):
        os.makedirs(db_dir, exist_ok=True)
        faiss.write_index(self.index, os.path.join(db_dir, "index.faiss"))
        with open(os.path.join(db_dir, "registry.json"), "w", encoding="utf-8") as fh:
            json.dump(self.registry, fh, ensure_ascii=False, indent=2)

    def ingest_pdf(self, path):
        if not path.lower().endswith(".pdf"):
            print("Pomijam (nie .pdf):", path)
            return 0

        pages = self._read_pdf(path)
        chunks = self.build_chunks(pages)
        self.index_chunks(chunks, source_name=os.path.basename(path))
        return len(chunks)

    def retrieve(self, query, top_k=10):
        query_vec = self.embedder.encode([query]).astype("float32")
        distances, ids = self.index.search(query_vec, top_k)

        results = []
        for idx in ids[0]:
            if idx == -1:
                continue
            results.append(self.registry[idx])
        return results

    def ask(self, prompt_template, query, llm_model, top_k=10, temperature=0.1, max_tokens=512):
        retrieved = self.retrieve(query, top_k=top_k)

        context_blocks = []
        for pos, item in enumerate(retrieved, start=1):
            header = f"[{pos}] {item['source']} (str. {item['page'] + 1})"
            context_blocks.append(f"{header}\n{item['text']}")

        full_context = "\n\n".join(context_blocks)
        prompt = prompt_template.format(context=full_context, query=query)

        reply = ollama.chat(
            model=llm_model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Bazuj wyłącznie na dostarczonym kontekście. "
                        "Nie dodawaj informacji spoza kontekstu. "
                        "Gdy kontekst nie wystarcza, odpowiedz: "
                        "Kontekst nie zawiera odpowiedzi na to pytanie."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            options={"temperature": temperature, "num_predict": max_tokens},
        )

        return reply["message"]["content"], retrieved

In [ ]:
indexer = DocumentIndexer(
    embedder=embedder,
    index=vector_index,
    registry=chunk_registry,
    window=900,
    overlap=150,
)

print("Indekser RAG gotowy do pracy")

In [ ]:
total_chunks = 0

for pdf_name in pdf_files:
    full_path = os.path.join(DOCS_DIR, pdf_name)
    print("Przetwarzam:", pdf_name)
    n = indexer.ingest_pdf(full_path)
    total_chunks += n
    print("  dodano fragmentów:", n)

indexer.save(DB_DIR)

print("Łączna liczba chunków w indeksie:", vector_index.ntotal)

In [ ]:
RAG_PROMPT = """
Masz dostęp do poniższego kontekstu źródłowego.

KONTEKST:
{context}

PYTANIE:
{query}

Zasady odpowiedzi:
- odpowiadaj wyłącznie po polsku,
- bądź zwięzły i konkretny,
- opieraj się tylko na podanym kontekście,
- jeśli kontekst nie zawiera odpowiedzi, napisz dokładnie:
  Kontekst nie zawiera odpowiedzi na to pytanie.
"""

In [ ]:
sample_question = "Kto potwierdza odbycie praktyki zawodowej w zakładzie pracy?"

reply_text, used_chunks = indexer.ask(
    prompt_template=RAG_PROMPT,
    query=sample_question,
    llm_model=LLM_NAME,
    top_k=10,
)

print("PYTANIE:", sample_question)
print()
print("ODPOWIEDŹ:")
print(reply_text)

In [ ]:
print("WYKORZYSTANE ŹRÓDŁA:")

for i, chunk in enumerate(used_chunks, start=1):
    print(f"\n--- Fragment {i} ---")
    print("Plik:", chunk["source"])
    print("Strona:", chunk["page"] + 1)
    print(chunk["text"][:700])

In [ ]:
test_questions = [
    "Ile godzin trwa praktyka zawodowa?",
    "Jakie dokumenty student musi złożyć po zakończeniu praktyki zawodowej?",
    "Co student powinien wiedzieć o technologiach i narzędziach stosowanych w informatyce?",
    "Wymień efekty uczenia się oznaczone numerami 01 02 03 które student musi osiągnąć w ramach praktyki zawodowej informatyka",
    "Kto sprawuje opiekę nad studentem ze strony uczelni podczas praktyki?",
    "Czy praktykę zawodową można odbyć w formie pracy zdalnej?",
]

for question in test_questions:
    answer, sources = indexer.ask(
        prompt_template=RAG_PROMPT,
        query=question,
        llm_model=LLM_NAME,
        top_k=10,
    )

    print("=" * 80)
    print("PYTANIE:", question)
    print("ODPOWIEDŹ:", answer)
    print("STRONY ŹRÓDŁOWE:", [c["page"] + 1 for c in sources[:3]])